# **Simulación de la Ventanilla Virtual para Optimizar la Cantidad de Operadores**

---

## **Descripción del Sistema**

La Ventanilla Virtual recibe consultas que el usuario clasifica en un tema/destinatario.
Cada tema se deriva a un departamento que atiende con su propio equipo y su propia cola:

- **GA (Gestión Académica):** Consultas académicas, inscripciones, etc.
- **A (Alumnos):** Consultas administrativas, documentación, etc.

**Horarios:** Lunes a viernes de 9 a 18 horas
**Objetivo:** Optimizar la cantidad de operadores por departamento para minimizar tiempos de atención sin exceder el presupuesto.

---

## **Variables del Modelo**

### **Variables Exógenas (No Controlables - Datos)**
- **IAGA:** Intervalo entre arribos en Gestión Académica (minutos)
- **IAA:** Intervalo entre arribos en Alumnos (minutos)
- **TAGA:** Tiempo de atención en Gestión Académica (minutos)
- **TAA:** Tiempo de atención en Alumnos (minutos)

### **Variables de Control**
- **N:** Puestos de atención de Gestión Académica
- **M:** Puestos de atención de Alumnos

### **Variables Endógenas**

**De Estado:**
- **NGA:** Cantidad de personas en la cola de Gestión Académica
- **NA:** Cantidad de personas en la cola de Alumnos

**De Resultado:**
- **PTOGA(n):** Porcentaje de tiempo ocioso de Gestión Académica
- **PTOA(m):** Porcentaje de tiempo ocioso de Alumnos
- **PRTGA:** Promedio de Respuesta por Ticket en Gestión Académica (minutos)
- **PRTA:** Promedio de Respuesta por Ticket en Alumnos (minutos)
- **PECGA:** Promedio de espera en cola de Gestión Académica (minutos)
- **PECA:** Promedio de espera en cola de Alumnos (minutos)

---

In [ ]:
import random
import math
import numpy as np
from scipy.special import gamma
import matplotlib.pyplot as plt

class SimuladorVentanillaVirtual:
    """
    Simulador de eventos discretos para el sistema de Ventanilla Virtual

    Simula el comportamiento de dos departamentos (GA y A) con sus respectivas
    colas y operadores, permitiendo evaluar diferentes configuraciones.
    """

    def __init__(self, N=2, M=2, tiempo_simulacion=9600):
        """
        Inicializa el simulador

        Args:
            N (int): Número de operadores en Gestión Académica (GA)
            M (int): Número de operadores en Alumnos (A)
            tiempo_simulacion (int): Tiempo total de simulación en minutos
        """
        # === PARÁMETROS DEL SISTEMA ===
        self.N = N  # Operadores GA
        self.M = M  # Operadores A
        self.TF = tiempo_simulacion  # Tiempo final de simulación
        self.HV = float('inf')  # Valor alto para "evento no programado"

        # === VARIABLES DE TIEMPO ===
        self.T = 0.0  # Tiempo actual de la simulación

        # === VARIABLES DE ESTADO ===
        self.NGA = 0  # Cantidad de clientes en cola GA
        self.NA = 0   # Cantidad de clientes en cola A

        # === CONTADORES TOTALES ===
        self.NTGA = 0  # Total de clientes procesados en GA
        self.NTA = 0   # Total de clientes procesados en A

        # === TIEMPOS DE EVENTOS PROGRAMADOS ===
        self.TPSGA = [self.HV] * N  # Tiempos programados de salida GA
        self.TPSA = [self.HV] * M   # Tiempos programados de salida A
        self.TPLLGA = 0.0           # Tiempo programado llegada GA
        self.TPLLA = 0.0            # Tiempo programado llegada A

        # === ACUMULADORES PARA ESTADÍSTICAS ===
        # Tiempos de salidas
        self.STSGA = 0.0  # Suma tiempos de sistema GA
        self.STSA = 0.0   # Suma tiempos de sistema A

        # Tiempos de llegada (para calcular tiempo en sistema)
        self.STLLGA = 0.0  # Suma tiempos de llegada GA
        self.STLLA = 0.0   # Suma tiempos de llegada A

        # Tiempos de atención/servicio
        self.STAGA = 0.0  # Suma tiempos de atención GA
        self.STAA = 0.0   # Suma tiempos de atención A

        # Tiempos ociosos por operador
        self.STOGA = [0.0] * N  # Suma tiempo ocioso GA por operador
        self.STOA = [0.0] * M   # Suma tiempo ocioso A por operador
        self.ITOGA = [0.0] * N  # Inicio tiempo ocioso GA por operador
        self.ITOA = [0.0] * M   # Inicio tiempo ocioso A por operador

        # Control de operadores que han trabajado
        self.operador_trabajo_ga = [False] * N  # Si el operador GA trabajó alguna vez
        self.operador_trabajo_a = [False] * M   # Si el operador A trabajó alguna vez

        # Listas para tracking detallado de tiempos en sistema
        self.tiempos_llegada_ga = []  # Tiempos de llegada individuales GA
        self.tiempos_llegada_a = []   # Tiempos de llegada individuales A

        # === PROGRAMAR PRIMERAS LLEGADAS ===
        self.TPLLGA = self.obtener_IAGA()  # Primera llegada GA
        self.TPLLA = self.obtener_IAA()    # Primera llegada A

        # Todos los operadores empiezan ociosos desde tiempo 0
        for i in range(N):
            self.ITOGA[i] = 0.0
        for i in range(M):
            self.ITOA[i] = 0.0

        print(f"=== INICIANDO SIMULACIÓN ===")
        print(f"Operadores GA: {N}, Operadores A: {M}")
        print(f"Tiempo de simulación: {tiempo_simulacion} minutos")
        print(f"Primera llegada GA: {self.TPLLGA:.2f}")
        print(f"Primera llegada A: {self.TPLLA:.2f}")
        print("=" * 50)

    # ==================== FUNCIONES DE DISTRIBUCIONES ====================

    def _foldcauchy_ppf(self, u, c, loc, scale, eps=1e-12):
        """
        Función de distribución inversa para Folded Cauchy
        Utilizada para generar intervalos entre arribos
        """
        u = max(eps, min(1.0 - eps, u))
        if abs(u - 0.5) <= 1e-12:
            z = math.sqrt(1.0 + c*c)
        else:
            t = math.tan(math.pi * u)
            s = 1.0 if t >= 0.0 else -1.0
            z = (-1.0 + s * math.sqrt(1.0 + (1.0 + c*c) * t*t)) / t
        return loc + scale * z

    def _gennorm_pdf(self, x, beta, loc, scale):
        """
        Función de densidad de probabilidad para Generalized Normal
        Utilizada en método de rechazo para tiempos de atención
        """
        z = np.abs((x - loc) / scale)
        return (beta / (2.0 * scale * gamma(1.0 / beta))) * np.exp(-np.power(z, beta))

    def _gennorm_M(self, beta, loc, scale, a, b):
        """Calcula la cota superior M para el método de rechazo"""
        if a <= loc <= b:
            return self._gennorm_pdf(loc, beta, loc, scale)
        else:
            return max(self._gennorm_pdf(a, beta, loc, scale),
                      self._gennorm_pdf(b, beta, loc, scale))

    def _rej_rect_uniform(self, f_pdf, a, b, M):
        """Método de rechazo rectangular uniforme"""
        while True:
            r1 = random.random()
            r2 = random.random()
            x = a + (b - a) * r1
            y = M * r2
            if y <= f_pdf(x):
                return x

    def obtener_IAA(self):
        """
        Genera intervalos entre arribos para Alumnos (A)
        Distribución: Folded Cauchy
        """
        R = random.uniform(0.001, 0.982)
        return self._foldcauchy_ppf(R, c=0.6257022419470324,
                                   loc=0.9999999933988453,
                                   scale=6.172313207823716)

    def obtener_IAGA(self):
        """
        Genera intervalos entre arribos para Gestión Académica (GA)
        Distribución: Folded Cauchy
        """
        R = random.uniform(0.001, 0.982)
        return self._foldcauchy_ppf(R, c=0.5561228593754299,
                                   loc=1.016666665640046,
                                   scale=5.788069654548192)

    def obtener_TAA(self):
        """
        Genera tiempos de atención para Alumnos (A)
        Distribución: Generalized Normal con método de rechazo
        """
        beta, loc, scale = 1.8331068455079633, 23.56547209570904, 6.634904447389176
        a, b = 10.0, 40.0
        f_pdf = lambda x: self._gennorm_pdf(x, beta, loc, scale)
        M = self._gennorm_M(beta, loc, scale, a, b)
        return self._rej_rect_uniform(f_pdf, a, b, M)

    def obtener_TAGA(self):
        """
        Genera tiempos de atención para Gestión Académica (GA)
        Distribución: Generalized Normal con método de rechazo
        """
        beta, loc, scale = 2.207237162715657, 12.476341431070377, 5.895534092625402
        a, b = 1.0, 30.0
        f_pdf = lambda x: self._gennorm_pdf(x, beta, loc, scale)
        M = self._gennorm_M(beta, loc, scale, a, b)
        return self._rej_rect_uniform(f_pdf, a, b, M)

    # ==================== FUNCIONES DE MANEJO DE EVENTOS ====================

    def obtener_Proximo_Evento(self):
        """
        Determina cuál es el próximo evento a procesar

        Returns:
            int: Tipo de evento (0: Salida GA, 1: Salida A, 2: Llegada A, 3: Llegada GA)
        """
        eventos = []

        # Recopilar todas las salidas programadas de GA
        for i, tiempo in enumerate(self.TPSGA):
            if tiempo < self.HV:
                eventos.append((0, tiempo, i))  # (tipo, tiempo, servidor)

        # Recopilar todas las salidas programadas de A
        for i, tiempo in enumerate(self.TPSA):
            if tiempo < self.HV:
                eventos.append((1, tiempo, i))  # (tipo, tiempo, servidor)

        # Agregar llegadas programadas
        if self.TPLLA < self.HV:
            eventos.append((2, self.TPLLA, -1))  # Llegada A
        if self.TPLLGA < self.HV:
            eventos.append((3, self.TPLLGA, -1))  # Llegada GA

        if not eventos:
            return None, self.HV, -1

        # Retornar el evento más próximo en tiempo
        tipo, tiempo, servidor = min(eventos, key=lambda x: x[1])
        return tipo

    def Persona_Libre(self, equipo):
        """
        Encuentra el primer operador libre en un equipo

        Args:
            equipo (list): Lista de tiempos programados de salida

        Returns:
            int: Índice del operador libre, o -1 si no hay ninguno
        """
        for i, tiempo in enumerate(equipo):
            if tiempo >= self.HV:
                return i
        return -1

    def obtener_tiempo_ocioso(self, T_actual, inicio_ocio):
        """
        Calcula el tiempo ocioso de un operador

        Args:
            T_actual (float): Tiempo actual
            inicio_ocio (float): Tiempo de inicio del período ocioso

        Returns:
            float: Tiempo ocioso transcurrido
        """
        if inicio_ocio < T_actual:
            return T_actual - inicio_ocio
        return 0.0

    # ==================== EVENTOS DEL SISTEMA ====================

    def salida_GA(self):
        """
        Procesa una salida (finalización de atención) en Gestión Académica

        Este evento ocurre cuando un operador de GA termina de atender a un cliente.
        """
        # Encontrar cuál operador tiene la salida más próxima
        tiempos_validos = [(i, t) for i, t in enumerate(self.TPSGA) if t < self.HV]
        if not tiempos_validos:
            return

        servidor, tiempo_salida = min(tiempos_validos, key=lambda x: x[1])

        #print(f"[{self.T:.2f}] Salida GA - Servidor {servidor}, Tiempo salida: {tiempo_salida:.2f}")

        # Actualizar tiempo actual
        self.T = tiempo_salida

        # Acumular tiempo de sistema
        self.STSGA += self.T

        # Un cliente menos en el sistema GA
        self.NGA -= 1

        if self.NGA >= self.N:
            # Hay cola: asignar siguiente cliente al operador que se libera
            TAGA = self.obtener_TAGA()
            self.TPSGA[servidor] = self.T + TAGA
            self.STAGA += TAGA
            #print(f"    Hay cola (NGA={self.NGA}). Próxima salida en {self.TPSGA[servidor]:.2f}")
        else:
            # No hay cola: operador queda libre
            self.ITOGA[servidor] = self.T  # Inicio período ocioso
            self.TPSGA[servidor] = self.HV
            #print(f"    No hay cola (NGA={self.NGA}). Servidor {servidor} libre")

    def salida_A(self):
        """
        Procesa una salida (finalización de atención) en Alumnos
        """
        # Encontrar cuál operador tiene la salida más próxima
        tiempos_validos = [(i, t) for i, t in enumerate(self.TPSA) if t < self.HV]
        if not tiempos_validos:
            return

        servidor, tiempo_salida = min(tiempos_validos, key=lambda x: x[1])

        #print(f"[{self.T:.2f}] Salida A - Servidor {servidor}, Tiempo salida: {tiempo_salida:.2f}")

        # Actualizar tiempo actual
        self.T = tiempo_salida

        # Acumular tiempo de sistema (CORREGIDO)
        self.STSA += self.T

        # Un cliente menos en el sistema A
        self.NA -= 1

        if self.NA >= self.M:
            # Hay cola: asignar siguiente cliente
            TAA = self.obtener_TAA()
            self.TPSA[servidor] = self.T + TAA
            self.STAA += TAA
            #print(f"    Hay cola (NA={self.NA}). Próxima salida en {self.TPSA[servidor]:.2f}")
        else:
            # No hay cola: operador queda libre
            self.ITOA[servidor] = self.T  # Inicio período ocioso
            self.TPSA[servidor] = self.HV
            #print(f"    No hay cola (NA={self.NA}). Servidor {servidor} libre")

    def llegada_GA(self):
        """
        Procesa una llegada a Gestión Académica

        Este evento ocurre cuando llega un nuevo cliente al departamento GA.
        """
        #print(f"[{self.T:.2f}] Llegada GA")

        # Acumular tiempo de llegada (para cálculo de tiempo en sistema)
        self.STLLGA += self.T

        # Actualizar tiempo actual al momento de la llegada
        self.T = self.TPLLGA

        # Guardar tiempo individual de llegada
        self.tiempos_llegada_ga.append(self.T)

        # Programar próxima llegada
        IAGA = self.obtener_IAGA()
        self.TPLLGA = self.T + IAGA

        # Incrementar contadores
        self.NGA += 1
        self.NTGA += 1

        #print(f"    Cliente {self.NTGA} llega. NGA={self.NGA}. Próxima llegada: {self.TPLLGA:.2f}")

        if self.NGA <= self.N:
            # Hay operador disponible
            operador_libre = self.Persona_Libre(self.TPSGA)
            if operador_libre >= 0:
                # Calcular y acumular tiempo ocioso del operador
                tiempo_ocioso = self.obtener_tiempo_ocioso(self.T, self.ITOGA[operador_libre])
                self.STOGA[operador_libre] += tiempo_ocioso

                # Marcar que este operador trabajó
                self.operador_trabajo_ga[operador_libre] = True

                # Asignar cliente al operador
                TAGA = self.obtener_TAGA()
                self.TPSGA[operador_libre] = self.T + TAGA
                self.STAGA += TAGA

                #print(f"    Asignado a operador {operador_libre}. Fin atención: {self.TPSGA[operador_libre]:.2f}")
                #print(f"    Tiempo ocioso operador {operador_libre}: {tiempo_ocioso:.2f}")
        # else:
        #     print(f"    No hay operadores libres. Cliente entra en cola.")

    def llegada_A(self):
        """
        Procesa una llegada a Alumnos
        """
        #print(f"[{self.T:.2f}] Llegada A")

        # Acumular tiempo de llegada
        self.STLLA += self.T

        # Actualizar tiempo actual
        self.T = self.TPLLA

        # Guardar tiempo individual de llegada
        self.tiempos_llegada_a.append(self.T)

        # Programar próxima llegada
        IAA = self.obtener_IAA()
        self.TPLLA = self.T + IAA

        # Incrementar contadores
        self.NA += 1
        self.NTA += 1

        #print(f"    Cliente {self.NTA} llega. NA={self.NA}. Próxima llegada: {self.TPLLA:.2f}")

        if self.NA <= self.M:
            # Hay operador disponible
            operador_libre = self.Persona_Libre(self.TPSA)
            if operador_libre >= 0:
                # Calcular y acumular tiempo ocioso
                tiempo_ocioso = self.obtener_tiempo_ocioso(self.T, self.ITOA[operador_libre])
                self.STOA[operador_libre] += tiempo_ocioso

                # Marcar que este operador trabajó
                self.operador_trabajo_a[operador_libre] = True

                # Asignar cliente al operador
                TAA = self.obtener_TAA()
                self.TPSA[operador_libre] = self.T + TAA
                self.STAA += TAA

                #print(f"    Asignado a operador {operador_libre}. Fin atención: {self.TPSA[operador_libre]:.2f}")
                #print(f"    Tiempo ocioso operador {operador_libre}: {tiempo_ocioso:.2f}")
        # else:
        #     print(f"    No hay operadores libres. Cliente entra en cola.")

    # ==================== SIMULACIÓN PRINCIPAL ====================

    def simular(self, debug=False, max_eventos_debug=100):
        """
        Ejecuta la simulación de eventos discretos

        Args:
            debug (bool): Si mostrar información detallada de eventos
            max_eventos_debug (int): Máximo número de eventos a mostrar en debug
        """
        eventos_procesados = 0

        print(f"\n=== INICIANDO FASE DE SIMULACIÓN ===")

        # Fase principal: simular hasta tiempo final
        while self.T < self.TF:
            tipo_evento = self.obtener_Proximo_Evento()

            if tipo_evento is None:
                #print("No hay más eventos programados")
                break

            if debug and eventos_procesados < max_eventos_debug:
                print(f"\nEvento {eventos_procesados + 1}: Tipo {tipo_evento}")

            # Procesar evento según su tipo
            if tipo_evento == 0:    # Salida GA
                self.salida_GA()
            elif tipo_evento == 1:  # Salida A
                self.salida_A()
            elif tipo_evento == 2:  # Llegada A
                self.llegada_A()
            elif tipo_evento == 3:  # Llegada GA
                self.llegada_GA()

            eventos_procesados += 1

            if debug and eventos_procesados < max_eventos_debug:
                print(f"    T={self.T:.2f}, NGA={self.NGA}, NA={self.NA}")

            # Verificar condición de parada
            if self.T >= self.TF:
                print(f"\n=== TIEMPO FINAL ALCANZADO (T={self.T:.2f}) ===")
                break

        # Fase de vaciamiento: procesar clientes restantes
        print(f"\n=== INICIANDO FASE DE VACIAMIENTO ===")
        print(f"Clientes restantes - GA: {self.NGA}, A: {self.NA}")

        # No programar más llegadas
        self.TPLLGA = self.HV
        self.TPLLA = self.HV

        while self.NGA > 0 or self.NA > 0:
            tipo_evento = self.obtener_Proximo_Evento()

            if tipo_evento is None:
                #print("No hay más salidas programadas")
                break

            if tipo_evento == 0:  # Solo salidas GA
                self.salida_GA()
            elif tipo_evento == 1:  # Solo salidas A
                self.salida_A()

            eventos_procesados += 1

        print(f"\n=== SIMULACIÓN FINALIZADA ===")
        print(f"Eventos procesados: {eventos_procesados}")
        print(f"Tiempo final: {self.T:.2f} minutos")

        return eventos_procesados

    def calcular_resultados(self):
        """
        Calcula las métricas finales del sistema según las fórmulas especificadas

        Fórmulas utilizadas:
        - PTOGA: (STOGA * 100) / TF  # CORREGIDO: usar TF, no T
        - PTOA: (STOA * 100) / TF   # CORREGIDO: usar TF, no T
        - PRTGA: (STSGA - STLLGA) / NTGA
        - PRTA: (STSA - STLLA) / NTA
        - PECGA: (STSGA - STLLGA - STAGA) / NTGA
        - PECA: (STSA - STLLA - STAA) / NTA
        """

        # === FINALIZAR CÁLCULO DE TIEMPO OCIOSO ===
        for i in range(self.N):
            if self.TPSGA[i] >= self.HV:  # Operador libre al final
                if self.operador_trabajo_ga[i]:  # Operador que trabajó en algún momento
                    if self.ITOGA[i] < self.TF:
                        tiempo_ocioso_final = self.TF - self.ITOGA[i]
                        self.STOGA[i] += tiempo_ocioso_final
                else:  # Operador que NUNCA trabajó
                    self.STOGA[i] = self.TF  # TODO el tiempo fue ocioso

        for i in range(self.M):
            if self.TPSA[i] >= self.HV:  # Operador libre al final
                if self.operador_trabajo_a[i]:  # Operador que trabajó en algún momento
                    if self.ITOA[i] < self.TF:
                        tiempo_ocioso_final = self.TF - self.ITOA[i]
                        self.STOA[i] += tiempo_ocioso_final
                else:  # Operador que NUNCA trabajó
                    self.STOA[i] = self.TF  # TODO el tiempo fue ocioso

        print("\n" + "="*80)
        print("RESULTADOS DE LA SIMULACIÓN DE VENTANILLA VIRTUAL")
        print("="*80)
        print(f"Configuración: {self.N} operadores GA, {self.M} operadores A")
        print(f"Tiempo de simulación: {self.TF} minutos ({self.TF/60:.1f} horas)")
        print(f"Tiempo final real: {self.T:.2f} minutos")

        # === ESTADÍSTICAS GENERALES ===
        print(f" ESTADÍSTICAS GENERALES:")
        print(f"   Total clientes GA procesados (NTGA): {self.NTGA}")
        print(f"   Total clientes A procesados (NTA): {self.NTA}")
        print(f"   Total eventos procesados: {self.NTGA + self.NTA}")

        # === TIEMPOS PROMEDIO GESTIÓN ACADÉMICA ===
        if self.NTGA > 0:
            # Fórmulas según especificación
            PRTGA = (self.STSGA - self.STLLGA) / self.NTGA  # Tiempo promedio de respuesta
            PECGA = (self.STSGA - self.STLLGA - self.STAGA) / self.NTGA  # Tiempo promedio de espera
            tiempo_servicio_prom_ga = self.STAGA / self.NTGA  # Tiempo promedio de servicio

            print(f" GESTIÓN ACADÉMICA (GA):")
            print(f"   Tiempo promedio de respuesta (PRTGA): {PRTGA:.2f} minutos")
            print(f"   Tiempo promedio de espera en cola (PECGA): {PECGA:.2f} minutos")
            print(f"   Tiempo promedio de servicio: {tiempo_servicio_prom_ga:.2f} minutos")

        # === TIEMPOS PROMEDIO ALUMNOS ===
        if self.NTA > 0:
            # Fórmulas según especificación
            PRTA = (self.STSA - self.STLLA) / self.NTA  # Tiempo promedio de respuesta
            PECA = (self.STSA - self.STLLA - self.STAA) / self.NTA  # Tiempo promedio de espera
            tiempo_servicio_prom_a = self.STAA / self.NTA  # Tiempo promedio de servicio

            print(f" ALUMNOS (A):")
            print(f"   Tiempo promedio de respuesta (PRTA): {PRTA:.2f} minutos")
            print(f"   Tiempo promedio de espera en cola (PECA): {PECA:.2f} minutos")
            print(f"   Tiempo promedio de servicio: {tiempo_servicio_prom_a:.2f} minutos")

        # === UTILIZACIÓN DE OPERADORES ===
        print(f"\n⚡ UTILIZACIÓN DE OPERADORES:")

        # Gestión Académica
        print(f"   Gestión Académica:")
        for i in range(self.N):
            PTOGA_i = (self.STOGA[i] * 100) / self.TF
            utilizacion = max(0.0, 100 - PTOGA_i)
            status = " (nunca trabajó)" if not self.operador_trabajo_ga[i] else ""
            print(f"      Operador GA-{i+1}: Utilización {utilizacion:.1f}% | Ocioso {PTOGA_i:.1f}%{status}")

        # Alumnos
        print(f"   Alumnos:")
        for i in range(self.M):
            PTOA_i = (self.STOA[i] * 100) / self.TF
            utilizacion = max(0.0, 100 - PTOA_i)
            status = " (nunca trabajó)" if not self.operador_trabajo_a[i] else ""
            print(f"      Operador A-{i+1}: Utilización {utilizacion:.1f}% | Ocioso {PTOA_i:.1f}%{status}")

        print("="*80)

def ejecutar_experimento(configuraciones, tiempo_sim=1440, debug=False):
    """
    Ejecuta múltiples configuraciones del sistema para comparar resultados

    Args:
        configuraciones (list): Lista de tuplas (N, M) con operadores GA y A
        tiempo_sim (int): Tiempo de simulación en minutos
        debug (bool): Si mostrar información detallada
    """
    resultados = []

    print("INICIANDO EXPERIMENTO DE OPTIMIZACIÓN")
    print(f"Configuraciones a evaluar: {len(configuraciones)}")
    print(f"Tiempo por simulación: {tiempo_sim} minutos\n")

    for i, (N, M) in enumerate(configuraciones):
        print(f"\n{'='*60}")
        print(f"EXPERIMENTO {i+1}/{len(configuraciones)}: {N} operadores GA, {M} operadores A")
        print(f"{'='*60}")

        # Crear y ejecutar simulación
        sim = SimuladorVentanillaVirtual(N=N, M=M, tiempo_simulacion=tiempo_sim)
        eventos = sim.simular(debug=debug, max_eventos_debug=20 if debug else 0)
        sim.calcular_resultados()

        # Guardar resultados para comparación
        if sim.NTGA > 0 and sim.NTA > 0:
            PRTGA = (sim.STSGA - sim.STLLGA) / sim.NTGA
            PRTA = (sim.STSA - sim.STLLA) / sim.NTA
            PECGA = (sim.STSGA - sim.STLLGA - sim.STAGA) / sim.NTGA
            PECA = (sim.STSA - sim.STLLA - sim.STAA) / sim.NTA

            # Utilización promedio
            util_ga = np.mean([max(0.0, 100 - (sim.STOGA[j] * 100) / sim.TF) for j in range(N)])
            util_a = np.mean([max(0.0, 100 - (sim.STOA[j] * 100) / sim.TF) for j in range(M)])

            resultados.append({
                'N': N, 'M': M,
                'PRTGA': PRTGA, 'PRTA': PRTA,
                'PECGA': PECGA, 'PECA': PECA,
                'UTIL_GA': util_ga, 'UTIL_A': util_a,
                'NTGA': sim.NTGA, 'NTA': sim.NTA,
                'eventos': eventos
            })

    # === COMPARACIÓN DE RESULTADOS ===
    if len(resultados) > 1:
        print(f"\n{'='*80}")
        print(" COMPARACIÓN DE CONFIGURACIONES")
        print(f"{'='*80}")

        print(f"{'Config':<10} {'PRTGA':<8} {'PRTA':<8} {'PECGA':<8} {'PECA':<8} {'Util_GA':<8} {'Util_A':<8}")
        print("-" * 70)

        for r in resultados:
            print(f"{r['N']},{r['M']:<8} {r['PRTGA']:<8.1f} {r['PRTA']:<8.1f} {r['PECGA']:<8.1f} " +
                  f"{r['PECA']:<8.1f} {r['UTIL_GA']:<8.1f}% {r['UTIL_A']:<8.1f}%")

    return resultados

# ==================== EJEMPLO DE USO ====================

if __name__ == "__main__":

    print(" SIMULACIÓN DE VENTANILLA VIRTUAL - TP6")
    print("Optimización de operadores por departamento\n")

    # === EXPERIMENTO DE OPTIMIZACIÓN ===
    print("\n\n EXPERIMENTO DE OPTIMIZACIÓN")
    configuraciones = [
        (1, 1),  # Configuración mínima
        (1, 2),
        (2, 1),
        (2, 2),  # Configuración actual
        (3, 2),  # Más operadores GA
        (2, 3),  # Más operadores A
        (3, 3),  # Configuración máxima
    ]

    resultados = ejecutar_experimento(configuraciones, tiempo_sim=40*4*4*60, debug=False)

 SIMULACIÓN DE VENTANILLA VIRTUAL - TP6
Optimización de operadores por departamento

 SIMULACIÓN CON CONFIGURACIÓN BASE


 EXPERIMENTO DE OPTIMIZACIÓN
INICIANDO EXPERIMENTO DE OPTIMIZACIÓN
Configuraciones a evaluar: 7
Tiempo por simulación: 38400 minutos


EXPERIMENTO 1/7: 1 operadores GA, 1 operadores A
=== INICIANDO SIMULACIÓN ===
Operadores GA: 1, Operadores A: 1
Tiempo de simulación: 38400 minutos
Primera llegada GA: 8.01
Primera llegada A: 14.02

=== INICIANDO FASE DE SIMULACIÓN ===

=== TIEMPO FINAL ALCANZADO (T=38404.23) ===

=== INICIANDO FASE DE VACIAMIENTO ===
Clientes restantes - GA: 0, A: 685

=== SIMULACIÓN FINALIZADA ===
Eventos procesados: 10080
Tiempo final: 54665.56 minutos

RESULTADOS DE LA SIMULACIÓN DE VENTANILLA VIRTUAL
Configuración: 1 operadores GA, 1 operadores A
Tiempo de simulación: 38400 minutos (640.0 horas)
Tiempo final real: 54665.56 minutos
 ESTADÍSTICAS GENERALES:
   Total clientes GA procesados (NTGA): 2737
   Total clientes A procesados (NTA): 2303
   